In [1]:
import argparse
import os
import sys
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
from tqdm import tqdm
import warnings

import wfdb, ast # for ptb xl includes

warnings.filterwarnings("ignore")

In [2]:
def parse_args():
    defaults = {
        "input_dir": "../../data/",
        "output_dir": "../../data/preprocessed",
        "target_length": 1000,
        "sampling_rate": 500, # ptb xl has either 100 or 500
        "overwrite": False
    }
    return defaults

In [3]:
def resize_data_to_numpy(data, target_length):
    original_length = data.shape[0]

    if original_length < 2:
        return None

    if original_length == target_length:
        return data.astype(np.float32)

    x_old = np.linspace(0, 1, original_length)
    x_new = np.linspace(0, 1, target_length)

    f = interp1d(x_old, data, axis=0, kind="linear")
    data_resized = f(x_new)
    return data_resized.astype(np.float32)

In [4]:
def main():
    args = parse_args()

    # --- SETUP ---
    if not os.path.exists(args["input_dir"]):
        print(f"Error: The input directroy '{args['input_dir']}' does not exsist.")
        sys.exit(1)

    print(f"Starting preprocessing...")
    print(f"Input: {args['input_dir']}")
    print(f"Output: {args['output_dir']}")

    os.makedirs(args["output_dir"], exist_ok=True)

    # --- LOAD AND PROCESS METADATA ---
    print("Loading PTB-XL metadata...")
    Y = pd.read_csv(os.path.join(args["input_dir"], 'ptbxl_database.csv'), index_col='ecg_id')
    Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

    agg_df = pd.read_csv(os.path.join(args["input_dir"], 'scp_statements.csv'), index_col=0)
    agg_df = agg_df[agg_df.diagnostic == 1]

    def aggregate_diagnostic(y_dic):
        tmp = []
        for key in y_dic.keys():
            if key in agg_df.index:
                tmp.append(agg_df.loc[key].diagnostic_class)
        return list(set(tmp))

    Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

    Y_filtered = Y[Y['diagnostic_superclass'].map(len) == 1].copy()
    Y_filtered['label'] = Y_filtered['diagnostic_superclass'].apply(lambda x: x[0])

    print(f"Filtered dataset from {len(Y)} to {len(Y_filtered)} single-label records for cleaner clustering.")

    labels_path = os.path.join(args["output_dir"], "cleaned_labels.csv")
    Y_filtered[['filename_lr', 'filename_hr', 'label', 'strat_fold']].to_csv(labels_path)
    print(f"Saved cleaned metadata to {labels_path}")
    print("-" * 50)

    # --- PREPROCESS SIGNALS ---
    processed_count = 0
    skipped_count = 0
    error_count = 0

    for ecg_id, row in tqdm(Y_filtered.iterrows(), total=len(Y_filtered), desc="Preprocessing Signals"):
        rel_file_path = row['filename_lr'] if args["sampling_rate"] == 100 else row['filename_hr']
        full_file_path = os.path.join(args["input_dir"], rel_file_path)
        
        save_path = os.path.join(args["output_dir"], f"{ecg_id:05d}.npy")

        if os.path.exists(save_path) and not args["overwrite"]:
            skipped_count += 1
            continue

        try:
            signal, meta = wfdb.rdsamp(full_file_path)
            
            if np.isnan(signal).any():
                signal = np.nan_to_num(signal, nan=0.0)

            processed_data = resize_data_to_numpy(signal, args["target_length"])

            if processed_data is None:
                error_count += 1
                continue

            np.save(save_path, processed_data)
            processed_count += 1

        except Exception as e:
            print(f"Error processing ecg_id {ecg_id}: {e}")
            error_count += 1

    print("-" * 50)
    print(f"DONE!")
    print(f"Processed: {processed_count}")
    print(f"Skipped: {skipped_count}")
    print(f"Errors: {error_count}")

In [5]:
main()

Starting preprocessing...
Input: ../../data/
Output: ../../data/preprocessed
Loading PTB-XL metadata...
Filtered dataset from 21837 to 16272 single-label records for cleaner clustering.
Saved cleaned metadata to ../../data/preprocessed\cleaned_labels.csv
--------------------------------------------------


Preprocessing Signals: 100%|████████████████████████████████████████████████████| 16272/16272 [02:28<00:00, 109.89it/s]

--------------------------------------------------
DONE!
Processed: 16272
Skipped: 0
Errors: 0
